In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1: MONTHLY DISTRIBUTION OF REAL CLASSES
# ═══════════════════════════════════════════════════════════════════════════════

INPUT_FILE = "BBSWA_All_Merged.xlsx"
MONTH_COL  = "Month"
LABEL_COL  = "label"

print("=" * 65)
print("MONTHLY DISTRIBUTION OF CLASSES (REAL DATA)")
print("=" * 65)

# Load Data
df = pd.read_excel(INPUT_FILE).dropna(subset=[LABEL_COL, MONTH_COL])
df[LABEL_COL] = df[LABEL_COL].astype(float)

# Create a clean cross-tabulation table
dist_table = pd.crosstab(df[MONTH_COL], df[LABEL_COL], margins=True, margins_name="TOTAL")
print("\n[TABLE] Exact Observation Counts per Month:")
print("─" * 65)
print(dist_table)

# Plotting the Distribution
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

# Create crosstab without the 'Total' margin for the plot
dist_plot = pd.crosstab(df[MONTH_COL], df[LABEL_COL])
dist_plot.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='viridis', edgecolor='black')

plt.title("Distribution of Safety Observations per Month", fontsize=14, pad=15)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Number of Observations", fontsize=12)
plt.legend(title="Classes (0.0=Infra, 1.0=Obs, 1.5=Int, 2.0=Corr)", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
"""
Phase 1: Base Classification Model (Infrastructure vs. Human Risk)
==================================================================
This module trains and evaluates the Layer 1 model, which separates
Infrastructure/Environmental observations (0.0) from Human Risk (1.0+).

Evaluation is performed using two strategies:
  1. An 80/20 random split across the entire dataset.
  2. A Leave-One-Month-Out validation loop to test temporal generalization.

Outputs are saved as Excel files to be ingested by Phase 2.
"""

import re
import warnings
import numpy as np
import pandas as pd
from thefuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIGURATION & HYPERPARAMETERS
# =============================================================================

INPUT_FILE               = "BBSWA_All_Merged.xlsx"
MONTH_COL                = "Month"
OBSERVER_COL             = "Observer"
OBSERVATION_COL          = "Observationcomment"
LABEL_COL                = "label"

FUZZY_THRESHOLD          = 90
UNCLASSIFIED_WEIGHT      = 3.0
ML_DECISION_THRESHOLD    = 0.35
BEST_C_VALUE             = 2.0
SHORT_FILTER_ML_OVERRIDE = 0.40
ARABIC_CHAR_THRESHOLD    = 0.30
RANDOM_STATE             = 42
CV_FOLDS                 = 5

# =============================================================================
# 2. TEXT NORMALIZATION
# =============================================================================

_DIGIT_GLUE = re.compile(r'\d+([a-zàâäéèêëîïôùûüç])', re.IGNORECASE)
_ARABIC_RE  = re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]+')

def normalise(text: str) -> str:
    """Removes spaces between numbers and letters, and lowercases text."""
    return _DIGIT_GLUE.sub(r'\1', str(text).strip().lower())

def is_non_french(text: str) -> bool:
    """Detects if the text contains a significant portion of Arabic characters."""
    total_chars = len(text.replace(" ", ""))
    if total_chars == 0:
        return False
    return (sum(len(m) for m in _ARABIC_RE.findall(text)) / total_chars) >= ARABIC_CHAR_THRESHOLD

# =============================================================================
# 3. RULE DICTIONARIES
# =============================================================================

FUZZY_TARGETS = ["epi", "loto", "bavette", "casque", "gilet", "harnais"]

ROLE_PATTERNS = [
    r"\bquelqu'un[a-z]*\b", r"\bindividu[a-z]*\b", r"\bhomme[a-z]*\b",
    r"\bpreparateur[a-z]*\b", r"\bpréparateur[a-z]*\b", r"\bmanutentionnaire[a-z]*\b",
    r"\bconducteur[a-z]*\b", r"\bchauffeur[a-z]*\b", r"\bclark[a-z]*\b",
    r"\belectricien[a-z]*\b", r"\bélectricien[a-z]*\b", r"\bsoudeur[a-z]*\b",
    r"\bmaintenance\b", r"\bintervenant[a-z]*\b", r"\bouvrier[a-z]*\b",
    r"\bemployé[a-z]*\b", r"\bemploy[eé][a-z]*\b", r"\bsalarié[a-z]*\b",
    r"\bprestataire[a-z]*\b", r"\bvisiteur[a-z]*\b", r"\bchef[a-z]*\b",
    r"\bsuperviseur[a-z]*\b", r"\bmanager[a-z]*\b", r"\bresponsable[a-z]*\b",
    r"\bstagiaire[a-z]*\b", r"\bintérimaire[a-z]*\b", r"\binterimaire[a-z]*\b",
    r"\bpersonnel[a-z]*\b", r"\bcollègue[a-z]*\b", r"\bcollegu[a-z]*\b"
]

ACTION_PATTERNS = [
    r"\bsensibilis[a-zéèê]*\b", r"\baprès\s+sensibilis", r"\bsensibilisation\b",
    r"\bon\s+lui\s+a\b", r"\bnous\s+avons\b", r"\bon\s+a\b", r"\bbon\s+travail\b",
]

# =============================================================================
# 4. RULE ENGINES & ML PIPELINE
# =============================================================================

def check_fuzzy(text: str):
    for word in text.split():
        for target in FUZZY_TARGETS:
            if fuzz.ratio(word, target) >= FUZZY_THRESHOLD:
                return True, word, target
    return False, None, None

def check_regex_rules(text: str, patterns: list):
    for pat in patterns:
        if re.search(pat, text):
            return True, pat
    return False, None

def get_rule_block_info(obs: str):
    if is_non_french(obs):
        return "NonFrench", None, "arabic_script", True

    text = normalise(obs)
    hit, det, targ = check_fuzzy(text)

    if hit:
        return "Fuzzy_EntryGuard", 1.0, f"'{det}' (~{targ})", False
    if len(text.split()) <= 2:
        return "Short_Sentence", 0.0, "N/A", False

    hit, pat = check_regex_rules(text, ROLE_PATTERNS)
    if hit:
        return "Role_Rules", 1.0, pat, False

    hit, pat = check_regex_rules(text, ACTION_PATTERNS)
    if hit:
        return "Action_Rules", 1.0, pat, False

    return "Unclassified", None, "N/A", False

def build_ml_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, max_features=8000, sublinear_tf=True)),
        ("lr", LogisticRegression(C=BEST_C_VALUE, class_weight="balanced", max_iter=1000, solver="lbfgs", random_state=RANDOM_STATE)),
    ])

def run_pipeline(texts_series, model):
    results = []
    for obs in texts_series:
        block, rule_p, trigger, _ = get_rule_block_info(obs)
        text_norm = normalise(obs)
        ml_prob = model.predict_proba([text_norm])[0, 1]

        final_pred = rule_p
        if block == "Short_Sentence" and ml_prob >= SHORT_FILTER_ML_OVERRIDE:
            final_pred = None

        if final_pred is None:
            final_pred = 1.0 if ml_prob >= ML_DECISION_THRESHOLD else 0.0

        results.append({
            "Pred": final_pred,
            "Block": block,
            "Trigger": trigger,
            "ML_Prob": round(ml_prob, 4),
        })
    return pd.DataFrame(results)

# =============================================================================
# 5. EXECUTION: LOAD & PREPARE DATA
# =============================================================================

df_raw = pd.read_excel(INPUT_FILE).dropna(subset=[LABEL_COL, OBSERVATION_COL, MONTH_COL]).reset_index(drop=True)
df_raw["true_binary"] = df_raw[LABEL_COL].replace([1.5, 2.0], 1.0).astype(float)
df_raw["NonFrench"] = df_raw[OBSERVATION_COL].astype(str).apply(is_non_french)

df_clean = df_raw[~df_raw["NonFrench"]].copy().reset_index(drop=True)
months = sorted(df_clean[MONTH_COL].unique())

print(f"Detected {len(months)} Months: {months}\n")

# =============================================================================
# 6. RUN 1: 80/20 RANDOM SPLIT ON FULL DATASET
# =============================================================================

print("=" * 65)
print("RUN 1: 80/20 RANDOM SPLIT ON FULL DATASET")
print("=" * 65)

df_train_80, df_test_20 = train_test_split(
    df_clean, test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df_clean["true_binary"]
)

df_train_80 = df_train_80.reset_index(drop=True)
df_test_20  = df_test_20.reset_index(drop=True)

X_train_raw_80 = df_train_80[OBSERVATION_COL].astype(str)
y_train_80     = df_train_80["true_binary"]
X_test_raw_20  = df_test_20[OBSERVATION_COL].astype(str)
y_test_20      = df_test_20["true_binary"]

X_train_norm_80  = X_train_raw_80.apply(normalise)
train_blocks_80  = X_train_raw_80.apply(lambda x: get_rule_block_info(x)[0])
train_weights_80 = train_blocks_80.apply(lambda b: UNCLASSIFIED_WEIGHT if b == "Unclassified" else 1.0).values

ml_model_80 = build_ml_pipeline()
ml_model_80.fit(X_train_norm_80, y_train_80, lr__sample_weight=train_weights_80)

test_results_80 = run_pipeline(X_test_raw_20, ml_model_80)
y_pred_test_80  = test_results_80["Pred"].values

print(f"  Test size : {len(y_test_20)} samples (Random 20% holdout)")
print(classification_report(y_test_20, y_pred_test_80, target_names=["0.0 Infra", "1.0 Human Risk"], digits=4, zero_division=0))
print(f"  Accuracy  : {accuracy_score(y_test_20, y_pred_test_80):.4f}\n")

df_train_80["Final_Pred"] = run_pipeline(X_train_raw_80, ml_model_80)["Pred"].values
df_test_20["Final_Pred"]  = y_pred_test_80

df_train_80.to_excel("phase1_train_Random_80_20.xlsx", index=False)
df_test_20.to_excel("phase1_test_Random_80_20.xlsx", index=False)
print("Saved -> phase1_train_Random_80_20.xlsx | phase1_test_Random_80_20.xlsx")

# =============================================================================
# 7. RUNS 2+: LEAVE-ONE-MONTH-OUT VALIDATION
# =============================================================================

for test_month in months:
    print("\n" + "═" * 65)
    print(f"EVALUATION: TEST ON [{test_month}] | TRAIN ON REST")
    print("═" * 65)

    # Split Data
    df_train = df_clean[df_clean[MONTH_COL] != test_month].copy().reset_index(drop=True)
    df_test  = df_clean[df_clean[MONTH_COL] == test_month].copy().reset_index(drop=True)

    X_train_raw = df_train[OBSERVATION_COL].astype(str)
    y_train     = df_train["true_binary"]
    X_test_raw  = df_test[OBSERVATION_COL].astype(str)
    y_test      = df_test["true_binary"]

    # Train Base Model
    X_train_norm  = X_train_raw.apply(normalise)
    train_blocks  = X_train_raw.apply(lambda x: get_rule_block_info(x)[0])
    train_weights = train_blocks.apply(lambda b: UNCLASSIFIED_WEIGHT if b == "Unclassified" else 1.0).values

    ml_model = build_ml_pipeline()
    ml_model.fit(X_train_norm, y_train, lr__sample_weight=train_weights)

    # Evaluate on Test Month
    test_results = run_pipeline(X_test_raw, ml_model)
    y_pred_test  = test_results["Pred"].values

    print(f"  Test size : {len(y_test)} samples ({test_month})")
    print(classification_report(y_test, y_pred_test, target_names=["0.0 Infra", "1.0 Human Risk"], digits=4, zero_division=0))
    print(f"  Accuracy  : {accuracy_score(y_test, y_pred_test):.4f}\n")

    # Save outputs for Phase 2
    df_train["Final_Pred"] = run_pipeline(X_train_raw, ml_model)["Pred"].values
    df_test["Final_Pred"]  = y_pred_test

    train_out = f"phase1_train_{test_month}.xlsx"
    test_out  = f"phase1_test_{test_month}.xlsx"
    df_train.to_excel(train_out, index=False)
    df_test.to_excel(test_out, index=False)

    print(f"Saved -> {train_out} | {test_out}")

Detected 5 Months: ['Decembre', 'Février', 'Janvier', 'Mars', 'Novembre']

RUN 1: 80/20 RANDOM SPLIT ON FULL DATASET
  Test size : 224 samples (Random 20% holdout)
                precision    recall  f1-score   support

     0.0 Infra     0.9091    0.7547    0.8247        53
1.0 Human Risk     0.9278    0.9766    0.9516       171

      accuracy                         0.9241       224
     macro avg     0.9184    0.8657    0.8882       224
  weighted avg     0.9234    0.9241    0.9216       224

  Accuracy  : 0.9241

Saved -> phase1_train_Random_80_20.xlsx | phase1_test_Random_80_20.xlsx

═════════════════════════════════════════════════════════════════
EVALUATION: TEST ON [Decembre] | TRAIN ON REST
═════════════════════════════════════════════════════════════════
  Test size : 60 samples (Decembre)
                precision    recall  f1-score   support

     0.0 Infra     0.0000    0.0000    0.0000         0
1.0 Human Risk     1.0000    0.9833    0.9916        60

      accuracy   

In [ ]:
"""
Phase 2: Cascaded Binary Classifier (Interaction vs. Correction)
================================================================
This module ingests the outputs from Phase 1 and processes observations
flagged as "Human Risk" (1.0+) through a two-step hierarchical pipeline:
  - Split A: Differentiates pure Observation (1.0) from Action Taken (1.5+).
  - Split B: Differentiates Interaction (1.5) from Correction (2.0).

This script dynamically loops through all test scenarios (Random 80/20
and Leave-One-Month-Out) generated by Phase 1, training dedicated ML
models for each scenario to prevent data leakage and evaluate real-world drift.
"""

import re
import glob
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIGURATION & HYPERPARAMETERS
# =============================================================================

OBS_COL           = "Observationcomment"
LBL_COL           = "label"
PRED_COL          = "Final_Pred"
RANDOM_STATE      = 42
CV_FOLDS          = 5

# ML confidence threshold required to predict 2.0 instead of 1.5
SPLIT_B_THRESHOLD = 0.55

TFIDF_PARAMS = dict(
    analyzer     = "char_wb",
    ngram_range  = (2, 4),
    min_df       = 2,
    max_features = 6000,
    sublinear_tf = True,
)

# =============================================================================
# 2. TEXT NORMALIZATION
# =============================================================================

_DIGIT_GLUE = re.compile(r'\d+([a-zàâäéèêëîïôùûüç])', re.IGNORECASE)

def normalise(text: str) -> str:
    """Removes spaces between numbers and letters, and lowercases text."""
    return _DIGIT_GLUE.sub(r'\1', str(text).strip().lower())

# =============================================================================
# 3. RULE DICTIONARIES (SPLIT A & SPLIT B)
# =============================================================================

INTERACTION_PATTERNS_A = [
    r"\bsensibilis[a-zéèê]*\b", r"\baprès\s+sensibilis",
    r"\bon\s+lui\s+a\b", r"\bnous\s+avons\b",
    r"\bon\s+a\s+(dit|inform|demand|averti|rappel|expliqu|conseil)",
    r"\bavert[ia][a-z]*\b", r"\binformé[es]*\b", r"\bexpliqu[eéè][a-z]*\b",
    r"\bdiscut[eéè][a-z]*\b", r"\béchang[eéè][a-z]*\b", r"\bprévenu[es]*\b",
    r"\bconscientis[eéè][a-z]*\b", r"\bintervenu\b", r"\bcorrigé[es]*\b",
    r"\bstoppé[es]*\b", r"\barrêté[es]*\b", r"\bmis\s+en\s+place\b",
    r"\bremis[e]*\b", r"\brangé[es]*\b", r"\bréparé[es]*\b", r"\bfixé[es]*\b",
    r"\bdone\b", r"\brappelé[es]*\b", r"\bdemandé[es]*\b"
]

CORRECTION_PATTERNS_B = [
    r"\bcorrig[eéè][a-z]*\b", r"\bstoppé[a-z]*\b", r"\barrêt[eéè][a-z]*\b", r"\bmis\s+en\s+place\b",
    r"\bremis[a-z]*\b", r"\brangé[a-z]*\b", r"\bréparé[a-z]*\b", r"\bfixé[a-z]*\b", r"\béquipé[a-z]*\b",
    r"\bsécurisé[a-z]*\b", r"\beffectué[a-z]*\b", r"\bport[eéè][a-z]*\b", r"\bobliger[a-z]*\b",
    r"\bréalis[eéè][a-z]*\b", r"\bdone\b", r"\bchangé[a-z]*\b", r"\bmodif[a-z]*\b", r"\brepris[a-z]*\b",
    r"\benlev[eéè][a-z]*\b", r"\binstall[eéè][a-z]*\b", r"\bmis\s+(son|sa|les|le|un)\b",
]

INTERACTION_ONLY_PATTERNS_B = [
    r"\brefus[eéè][a-z]*\b", r"\bpas\s+fait\b", r"\bavertissement\s+verbal\b",
    r"\bsensibilis[eéè][a-z]*\s+uniquement\b", r"\bpromis\s+de\b", r"\bn'a\s+pas\b", r"\bsans\s+suite\b"
]

def split_a_rule(text: str):
    t = normalise(text)
    for pat in INTERACTION_PATTERNS_A:
        if re.search(pat, t):
            return 1, pat
    return None, None

def split_b_rule(text: str):
    t = normalise(text)
    for pat in CORRECTION_PATTERNS_B:
        if re.search(pat, t):
            return 1, pat
    for pat in INTERACTION_ONLY_PATTERNS_B:
        if re.search(pat, t):
            return 0, pat
    return None, None

# =============================================================================
# 4. ML PIPELINE BUILDERS & DATA PREP
# =============================================================================

def build_pipeline(class_weights_dict=None):
    return Pipeline([
        ("tfidf", TfidfVectorizer(**TFIDF_PARAMS)),
        ("lr", LogisticRegression(C=2.0, class_weight=class_weights_dict, max_iter=1000, solver="lbfgs", random_state=RANDOM_STATE)),
    ])

def build_pipeline_b(class_weights_dict=None):
    combined_features = FeatureUnion([
        ("char_tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2, max_features=3000, sublinear_tf=True)),
        ("word_tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 3), min_df=2, max_features=3000, sublinear_tf=True))
    ])
    return Pipeline([
        ("features", combined_features),
        ("lr", LogisticRegression(C=1.0, class_weight=class_weights_dict, max_iter=1000, solver="lbfgs", random_state=RANDOM_STATE)),
    ])

def prep_phase2(df):
    """Filters dataset to only include samples flagged as Human Risk in Phase 1."""
    df_filtered = df[(df[PRED_COL] == 1.0) & (df[LBL_COL].isin([1.0, 1.5, 2.0]))].copy().reset_index(drop=True)
    df_filtered["obs_norm"] = df_filtered[OBS_COL].astype(str).apply(normalise)
    df_filtered["split_a_label"] = (df_filtered[LBL_COL] != 1.0).astype(int)
    df_filtered["split_b_label"] = (df_filtered[LBL_COL] == 2.0).astype(int)
    return df_filtered

# =============================================================================
# 5. DYNAMIC INFERENCE ENGINE
# =============================================================================

def predict_split_a(texts, model_a):
    preds, methods = [], []
    for txt in texts:
        rule_p, _ = split_a_rule(txt)
        if rule_p is not None:
            preds.append(rule_p)
            methods.append("rule")
        else:
            preds.append(model_a.predict([normalise(txt)])[0])
            methods.append("model")
    return np.array(preds), methods

def predict_split_b(texts, model_b, threshold=SPLIT_B_THRESHOLD):
    preds, methods = [], []
    for txt in texts:
        rule_p, _ = split_b_rule(txt)
        if rule_p is not None:
            preds.append(rule_p)
            methods.append("rule")
        else:
            prob_corrected = model_b.predict_proba([normalise(txt)])[0, 1]
            if prob_corrected >= threshold:
                preds.append(1)
            else:
                preds.append(0)
            methods.append("model")
    return np.array(preds), methods

def predict_cascade(texts, model_a, model_b):
    final_labels, paths = [], []
    a_preds_all, a_methods_all = predict_split_a(texts, model_a)

    for txt, a_pred, a_method in zip(texts, a_preds_all, a_methods_all):
        if a_pred == 0:
            final_labels.append("1.0")
            paths.append(f"A({a_method})→1.0")
        else:
            b_pred_arr, b_methods_all = predict_split_b([txt], model_b)
            b_pred, b_method = b_pred_arr[0], b_methods_all[0]
            if b_pred == 0:
                final_labels.append("1.5")
                paths.append(f"A({a_method})→B({b_method})→1.5")
            else:
                final_labels.append("2.0")
                paths.append(f"A({a_method})→B({b_method})→2.0")
    return final_labels, paths

def get_full_system_label(row, model_a, model_b):
    if row[PRED_COL] == 0.0:
        return "0.0"
    labels, _ = predict_cascade([str(row[OBS_COL])], model_a, model_b)
    return labels[0]

def get_layer1_and_2_label(row, model_a):
    if row[PRED_COL] == 0.0:
        return "0.0"
    obs_text = str(row[OBS_COL])
    a_pred_arr, _ = predict_split_a([obs_text], model_a)

    if a_pred_arr[0] == 0:
        return "1.0"
    else:
        return "1.5+"

def map_true_to_3_classes(val):
    v = float(val)
    if v == 0.0: return "0.0"
    if v == 1.0: return "1.0"
    return "1.5+"

# =============================================================================
# 6. MULTI-SCENARIO EXECUTION LOOP
# =============================================================================

print("=" * 65)
print("PHASE 2 — DYNAMIC CASCADED BINARY CLASSIFIER")
print("Architecture: Split A (Observe vs Interact) → Split B (Interact vs Correct)")
print("=" * 65)

# Dynamically discover which test scenarios were generated in Phase 1
train_files = sorted(glob.glob("phase1_train_*.xlsx"))
test_scenarios = [f.replace("phase1_train_", "").replace(".xlsx", "") for f in train_files]

for scenario in test_scenarios:
    print("\n" + "█" * 75)
    print(f"█ PHASE 2 END-TO-END REPORT | TEST SCENARIO: {scenario.upper()}")
    print("█" * 75)

    # Load dynamic split data
    df_train_raw = pd.read_excel(f"phase1_train_{scenario}.xlsx")
    df_test_raw  = pd.read_excel(f"phase1_test_{scenario}.xlsx")

    train_df = prep_phase2(df_train_raw)
    test_df  = prep_phase2(df_test_raw)

    print(f"Phase 2 Train rows : {len(train_df)}")
    print(f"Phase 2 Test rows  : {len(test_df)}")

    # -------------------------------------------------------------------------
    # Train Split A Model
    # -------------------------------------------------------------------------
    X_a_train = train_df["obs_norm"].values
    y_a_train = train_df["split_a_label"].values
    X_a_test  = test_df["obs_norm"].values
    y_a_test  = test_df["split_a_label"].values

    cw_a_arr = compute_class_weight("balanced", classes=np.array([0,1]), y=y_a_train)
    model_a = build_pipeline({0: cw_a_arr[0], 1: cw_a_arr[1]})
    model_a.fit(X_a_train, y_a_train)

    # -------------------------------------------------------------------------
    # Train Split B Model
    # -------------------------------------------------------------------------
    train_b = train_df[train_df[LBL_COL].isin([1.5, 2.0])].reset_index(drop=True)
    test_b  = test_df[test_df[LBL_COL].isin([1.5, 2.0])].reset_index(drop=True)

    X_b_train = train_b["obs_norm"].values
    y_b_train = train_b["split_b_label"].values
    X_b_test  = test_b["obs_norm"].values
    y_b_test  = test_b["split_b_label"].values

    # Unweighted dictionary to allow the model to learn the natural distribution
    model_b = build_pipeline_b(class_weights_dict=None)
    model_b.fit(X_b_train, y_b_train)

    # -------------------------------------------------------------------------
    # Generate & Print Reports
    # -------------------------------------------------------------------------

    # 1. Split A Report
    print(f"\n[SPLIT A] Test Report ({scenario}):")
    a_preds, _ = predict_split_a(test_df[OBS_COL].astype(str).tolist(), model_a)
    print(classification_report(y_a_test, a_preds, labels=[0, 1], target_names=["0 — Observe (1.0)", "1 — Interact (1.5+2.0)"], digits=4, zero_division=0))

    # 2. Split B Report
    print(f"\n[SPLIT B] Test Report ({scenario}) - Threshold: {SPLIT_B_THRESHOLD}:")
    b_preds, _ = predict_split_b(test_b[OBS_COL].astype(str).tolist(), model_b)
    print(classification_report(y_b_test, b_preds, labels=[0, 1], target_names=["0 — Interacted (1.5)", "1 — Corrected (2.0)"], digits=4, zero_division=0))

    # 3. Full System Evaluation (4 Classes)
    print(f"\nREPORT D — FULL SYSTEM END-TO-END ({scenario})")
    print("─" * 65)

    df_test_raw["System_Label"]   = df_test_raw.apply(lambda row: get_full_system_label(row, model_a, model_b), axis=1)
    df_test_raw["true_label_str"] = df_test_raw[LBL_COL].astype(float).map({0.0: "0.0", 1.0: "1.0", 1.5: "1.5", 2.0: "2.0"})

    print(f"  Total samples in final test : {len(df_test_raw)}\n")
    print(classification_report(
        df_test_raw["true_label_str"], df_test_raw["System_Label"],
        labels=["0.0", "1.0", "1.5", "2.0"],
        target_names=["0.0 Infra", "1.0 Observed", "1.5 Interacted", "2.0 Corrected"],
        digits=4, zero_division=0
    ))

    final_acc = accuracy_score(df_test_raw['true_label_str'], df_test_raw['System_Label'])
    print(f"  Overall Accuracy : {final_acc:.4f}\n")

    out_file = f"final_safety_cascade_{scenario}.xlsx"
    df_test_raw.to_excel(out_file, index=False)

    # 4. Intermediate Report (3-Class Foundation Health)
    print(f"\nINTERMEDIATE REPORT — PHASE 1 + SPLIT A ONLY ({scenario})")
    print("─" * 65)

    df_test_raw["L1_L2_Pred"] = df_test_raw.apply(lambda row: get_layer1_and_2_label(row, model_a), axis=1)
    df_test_raw["L1_L2_True"] = df_test_raw[LBL_COL].apply(map_true_to_3_classes)

    labels_3class = ["0.0", "1.0", "1.5+"]
    print(classification_report(
        df_test_raw["L1_L2_True"], df_test_raw["L1_L2_Pred"],
        labels=labels_3class,
        target_names=["0.0 Infra", "1.0 Observed", "1.5+ Action Taken"],
        digits=4, zero_division=0
    ))

    acc_3 = accuracy_score(df_test_raw["L1_L2_True"], df_test_raw["L1_L2_Pred"])
    print(f"  3-Class Foundation Accuracy : {acc_3:.4f} ({acc_3*100:.2f}%)\n")

PHASE 2 — DYNAMIC CASCADED BINARY CLASSIFIER
Architecture: Split A (Observe vs Interact) → Split B (Interact vs Correct)

███████████████████████████████████████████████████████████████████████████
█ PHASE 2 END-TO-END REPORT | TEST SCENARIO: DECEMBRE
███████████████████████████████████████████████████████████████████████████
Phase 2 Train rows : 787
Phase 2 Test rows  : 59

[SPLIT A] Test Report (Decembre):
                        precision    recall  f1-score   support

     0 — Observe (1.0)     1.0000    0.5714    0.7273         7
1 — Interact (1.5+2.0)     0.9455    1.0000    0.9720        52

              accuracy                         0.9492        59
             macro avg     0.9727    0.7857    0.8496        59
          weighted avg     0.9519    0.9492    0.9429        59


[SPLIT B] Test Report (Decembre) - Threshold: 0.55:
                      precision    recall  f1-score   support

0 — Interacted (1.5)     1.0000    0.1400    0.2456        50
 1 — Corrected (2.0)   

In [ ]:
import re
import unicodedata
import pandas as pd
import glob
from thefuzz import fuzz
from collections import Counter

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4: MONTHLY OBSERVER LEADERBOARDS (REAL VS PREDICTED)
# Global cross-month observer deduplication — v3
# ═══════════════════════════════════════════════════════════════════════════════

# =============================================================================
# 1. NAME NORMALIZATION
# =============================================================================

# Matches pure system/functional entries with no real name attached:
#   "REGLQ1_2 (funct)" alone → drop row
#   "REGLQ1_2 (funct)ighit Saber" → keep "ighit Saber"
_SYSTEM_TAG_RE  = re.compile(
    r'^(REGL[A-Z0-9_]+\s*\([^)]+\)|#N/A|N/A|nan)\s*',
    re.IGNORECASE
)
_SITE_SUFFIX_RE = re.compile(
    r'\s*[A-Za-z]?\s*/\s*[Aa][Zz]\b|\s*[A-Za-z]?\s*/\s*[Zz]\b',
    re.IGNORECASE
)

# Manual overrides: force specific raw spellings to a chosen canonical.
# Use this when the frequency-wins rule picks a corrupted/ugly variant.
MANUAL_OVERRIDES = {
    'Abd Ellah Manaa/Az': 'Abdellah Manaa',
    'Khalad Rabatchi /Az': 'Rebatchi Khaled A/z',   # also catches word-swap
}


def strip_accents(text: str) -> str:
    return ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )


def is_ghost_entry(name: str) -> bool:
    """True if the name is a pure system tag with no real name content."""
    cleaned = _SYSTEM_TAG_RE.sub('', str(name).strip())
    return cleaned == ''


def clean_system_tag(name: str) -> str:
    cleaned = _SYSTEM_TAG_RE.sub('', str(name).strip())
    return cleaned if cleaned else name


def normalize_key(name: str) -> str:
    """
    Canonical sort key (never shown to user):
      strip system tag → strip site suffix → strip accents
      → lowercase → remove non-alpha → sort words alphabetically
    """
    name = clean_system_tag(str(name).strip())
    name = _SITE_SUFFIX_RE.sub('', name)
    name = strip_accents(name)
    name = re.sub(r'[^a-z\s]', '', name.lower())
    name = re.sub(r'\s+', ' ', name).strip()
    return ' '.join(sorted(name.split()))


def name_cleanliness(name: str) -> int:
    """
    Score for picking the 'cleanest' canonical when frequency is tied.
    Higher = cleaner. Penalizes: site suffix, system tag, space-in-name
    anomalies (Abd Ellah vs Abdellah), all-lowercase, word-swapped order.
    """
    score = 0
    if not _SITE_SUFFIX_RE.search(name):
        score += 2                          # no /Az suffix = cleaner
    if not _SYSTEM_TAG_RE.search(name):
        score += 2                          # no system tag
    if name[0].isupper():
        score += 1                          # properly capitalised
    if '  ' not in name:
        score += 1                          # no double spaces
    return score


# =============================================================================
# 2. GLOBAL REGISTRY
# =============================================================================

def build_global_registry(all_names: list,
                           fuzzy_threshold: int = 85) -> dict:
    """
    raw_name → canonical_raw_name, built across all months combined.

    Canonical selection priority:
      1. Manual overrides (MANUAL_OVERRIDES dict)
      2. Most frequent spelling across all months
      3. Tie-break: cleanliness score (no /Az, proper caps, etc.)
    """
    freq = Counter(all_names)
    norm_map = {n: normalize_key(n) for n in set(all_names)}

    # Group by exact normalized key
    groups: dict[str, list] = {}
    for raw, key in norm_map.items():
        groups.setdefault(key, []).append(raw)

    # Fuzzy-merge similar keys — catches single-letter typos like Khalad/Khaled
    keys = list(groups.keys())
    absorbed = set()
    for i, ka in enumerate(keys):
        if ka in absorbed:
            continue
        for kb in keys[i + 1:]:
            if kb in absorbed:
                continue
            if fuzz.ratio(ka, kb) >= fuzzy_threshold:
                groups[ka].extend(groups[kb])
                absorbed.add(kb)
    for k in absorbed:
        del groups[k]

    # Build registry: pick canonical per group
    registry: dict[str, str] = {}
    for members in groups.values():
        unique = list(set(members))

        # Check manual overrides first
        override = next(
            (MANUAL_OVERRIDES[m] for m in unique if m in MANUAL_OVERRIDES),
            None
        )
        if override:
            canonical = override
        else:
            # Frequency wins; tie-break by cleanliness
            canonical = max(
                unique,
                key=lambda n: (freq[n], name_cleanliness(n))
            )

        for raw in members:
            registry[raw] = canonical

    # Apply manual overrides as a final pass (catches cross-group remaps)
    for raw, canon in MANUAL_OVERRIDES.items():
        if raw in registry:
            registry[raw] = canon

    return registry


# =============================================================================
# 3. LOAD FILES & BUILD REGISTRY
# =============================================================================

output_files = sorted(glob.glob("final_safety_cascade_*.xlsx"))
monthly_files = [f for f in output_files if "Random_80_20" not in f]

if not monthly_files:
    print("No monthly output files found! Make sure you ran Cell 2.")
else:
    monthly_dfs: dict[str, pd.DataFrame] = {}
    for f in monthly_files:
        month = f.replace("final_safety_cascade_", "").replace(".xlsx", "")
        monthly_dfs[month] = pd.read_excel(f)

    # Pool all names across all months
    all_names = [
        n for df in monthly_dfs.values()
        for n in df['Observer'].dropna().tolist()
        if not is_ghost_entry(n)        # exclude pure system-tag rows globally
    ]

    GLOBAL_REGISTRY = build_global_registry(all_names, fuzzy_threshold=85)

    # Print audit log
    all_merges = sorted(
        set((r, c) for r, c in GLOBAL_REGISTRY.items() if r != c),
        key=lambda x: x[1]
    )
    print("=" * 85)
    print("GLOBAL NAME DEDUPLICATION REGISTRY")
    print(f"  {len(all_merges)} variant(s) unified across all months:")
    print("─" * 85)
    for raw, canon in all_merges:
        print(f"  '{raw}'  →  '{canon}'")
    print("=" * 85)

    # =============================================================================
    # 4. LEADERBOARD PER MONTH
    # =============================================================================

    print("\n" + "=" * 85)
    print("OBSERVER CLASSEMENT BY MONTH: REAL POINTS VS PREDICTED POINTS")
    print("=" * 85)

    for month_name, df in monthly_dfs.items():

        print("\n" + "█" * 85)
        print(f"█ LEADERBOARD | TEST MONTH: {month_name.upper()}")
        print("█" * 85)

        df = df.copy()

        # Drop ghost/system-only rows (e.g. bare "REGLQ1_2 (funct)")
        ghost_mask = df['Observer'].apply(
            lambda n: is_ghost_entry(str(n)) if pd.notna(n) else True
        )
        dropped_ghosts = df.loc[ghost_mask, 'Observer'].tolist()
        df = df[~ghost_mask].reset_index(drop=True)

        if dropped_ghosts:
            print(f"\n  [GHOST ROWS DROPPED — {len(dropped_ghosts)}]")
            for g in set(dropped_ghosts):
                print(f"    '{g}'")

        # Apply global registry
        month_merges = [
            (raw, canon)
            for raw, canon in GLOBAL_REGISTRY.items()
            if raw != canon and raw in df['Observer'].values
        ]
        df['Observer'] = df['Observer'].map(
            lambda n: GLOBAL_REGISTRY.get(n, n)
        )

        if month_merges:
            print(f"\n  [VARIANTS MERGED THIS MONTH — {len(month_merges)}]")
            for raw, canon in sorted(month_merges, key=lambda x: x[1]):
                print(f"    '{raw}'  →  '{canon}'")

        # Score & rank
        df['Real_Points'] = df['true_label_str'].astype(float)
        df['Pred_Points'] = df['System_Label'].astype(float)

        ranking = df.groupby('Observer')[['Real_Points', 'Pred_Points']].sum().reset_index()
        obs_count = df.groupby('Observer').size().reset_index(name='Total_Obs')
        ranking = pd.merge(ranking, obs_count, on='Observer')

        ranking['Real_Rank'] = ranking['Real_Points'].rank(
            ascending=False, method='min').astype(int)
        ranking['Pred_Rank'] = ranking['Pred_Points'].rank(
            ascending=False, method='min').astype(int)

        ranking['Rank_Shift'] = (ranking['Real_Rank'] - ranking['Pred_Rank']).apply(
            lambda x: f"▲ +{x}" if x > 0 else (f"▼ {x}" if x < 0 else "  -")
        )
        ranking['Point_Diff'] = (ranking['Pred_Points'] - ranking['Real_Points']).apply(
            lambda x: f"+{x:.1f}" if x > 0 else (f"{x:.1f}" if x < 0 else " 0.0")
        )

        ranking = ranking.sort_values('Real_Rank').reset_index(drop=True)
        final_display = ranking[['Observer', 'Total_Obs', 'Real_Points', 'Pred_Points',
                                  'Point_Diff', 'Real_Rank', 'Pred_Rank', 'Rank_Shift']]

        print(f"\n[TOP 15 OBSERVERS - {month_name.upper()}]")
        print("─" * 85)
        print(final_display.head(15).to_string(index=False))

        total_real = ranking['Real_Points'].sum()
        total_pred = ranking['Pred_Points'].sum()
        error_margin = (abs(total_real - total_pred) / total_real * 100) if total_real > 0 else 0

        print("\n  " + "-" * 45)
        print(f"  Monthly Real Points Awarded : {total_real:.1f}")
        print(f"  Monthly Pred Points Awarded : {total_pred:.1f}")
        print(f"  Monthly Point Error         : {error_margin:.2f}%")
        print("  " + "-" * 45 + "\n")

        final_display.to_excel(f"leaderboard_{month_name}.xlsx", index=False)